In [ ]:
import statsmodels.formula.api as smf

In [ ]:
# Create a nested grouping variable if needed (optional if you want nested random effects separately)
df['experiment_UID'] = df['experiment'].astype(str) + '_' + df['UID'].astype(str)
df['perc_compaction'] = df['% compaction']

# Fit the mixed model
model = smf.mixedlm(
    "perc_compaction ~ tx * t",  # fixed effects
    data=df,
    groups=df["experiment_UID"],  # random intercept for cell
    vc_formula={"experiment": "0 + C(experiment)"}  # random effect for biological replicate
)

result = model.fit()

# Output results
print(result.summary())

# Melt the dataframe
df_long = ctrl_df.melt(
    id_vars=['UID', 'experiment', 'tx', 'elapsed time (hr)'],
    value_vars=['LifeAct_intensity', '', 'mean actin intensity (compact)'],
    var_name='measurement_type',
    value_name='value'
)

# For each measurement type
for mtype in df_long['measurement_type'].unique():
    df_sub = df_long[df_long['measurement_type'] == mtype]

    model = smf.mixedlm("value ~ group * t", data=df_sub, groups=df_sub["biological_replicate"])
    result = model.fit()
    print(f"Result for {mtype}:")
    print(result.summary())